In [ ]:
from datetime import date, timedelta
import requests
import pandas as pd
import time


API_KEY = ""
headers = {"X-eBirdApiToken": API_KEY}
results = [] 

def get_data(Region, Date):
    # gets eBird data for a region and a date
    URL = f"https://api.ebird.org/v2/data/obs/{Region}/historic/{Date}"
    
    parameters = {
        "speciesCode": "rocpig",
        "includeProvisional": "true",
        "hotspot": "false"
    }

    
    print(f" start request for {Region} on the {Date}...")     
    response = requests.get(URL, headers=headers, params=parameters)

    # Check if the request was successful
    if response.status_code == 200:     
        data = response.json()

        # § LLM Help to filter the pigeons and getting the pigeon count of the day
        pigeons = [bird for bird in data if bird.get('speciesCode') == "rocpig"]
        pigeon_count = sum(
            bird.get("howMany", 1) for bird in pigeons
        )
        
        print(f"{pigeon_count} pigeon observations found.")

        # Store the aggregated results for export to CSV
        results.append({       
            "Date": Date,
            "Region": Region,
            "PigeonCount": pigeon_count
        })
    
    else:
        print(f"Error message: {response.status_code}")
        print(response.text)

# Define timeframe for the analysis
start_date = date(2020,1,1)     
end_date = date(2025,1,1)
total_days = (end_date - start_date).days

# Iterate through every day in the time range
for i in range(total_days + 1):
    # § LLM Help to set up the dates in the right format
    current_date = start_date + timedelta(days=i)
    formatted_date = current_date.strftime("%Y/%m/%d")

    get_data("DE-BE", formatted_date)

    # short pause to not overload the API
    time.sleep(0.2) 

# Setting up DataFrame and saving results into csv file
df = pd.DataFrame(results)    
df.to_csv("pigeons_in_berlin.csv", index=False)

print("Data saved as: pigeons_in_berlin.csv")
print(df.head())        #Test showing the first rows
print("-------DONE-------")


 start request for DE-BE on the 2020/01/01...
40 pigeon observations found.
 start request for DE-BE on the 2020/01/02...
1 pigeon observations found.
 start request for DE-BE on the 2020/01/03...
1 pigeon observations found.
 start request for DE-BE on the 2020/01/04...
50 pigeon observations found.
 start request for DE-BE on the 2020/01/05...
25 pigeon observations found.
 start request for DE-BE on the 2020/01/06...
0 pigeon observations found.
 start request for DE-BE on the 2020/01/07...
0 pigeon observations found.
 start request for DE-BE on the 2020/01/08...
0 pigeon observations found.
 start request for DE-BE on the 2020/01/09...
0 pigeon observations found.
 start request for DE-BE on the 2020/01/10...
0 pigeon observations found.
 start request for DE-BE on the 2020/01/11...
2 pigeon observations found.
 start request for DE-BE on the 2020/01/12...
0 pigeon observations found.
 start request for DE-BE on the 2020/01/13...
0 pigeon observations found.
 start request for DE-

In [ ]:
# weather API
from datetime import datetime
import requests
import pandas as pd
import time


API_KEY = ""
LAT = 52.52
LONG = 13.405
REGION = "Berlin"

air_pollution_results = []

def get_air_pollution_data(start_time, end_time):
    # get the data for air pollution for a time period from OpenWheater
    URL = "http://api.openweathermap.org/data/2.5/air_pollution/history"

    # API parameters coordinates, time window, and API key
    parameters = {
        "lat": LAT,
        "lon": LONG,
        "start": start_time,
        "end": end_time,
        "appid": API_KEY
    }

    response = requests.get(URL, params=parameters)

    # 1. case: everything runs smoothly
    if response.status_code == 200:
        data = response.json()

        # Iterate through the hourly data from the response list
        for entry in data.get("list", []):

            # § LLM Help for converting the timestamp to string
            dt = datetime.fromtimestamp(entry["dt"]).strftime("%Y-%m-%d")
            air_values = entry["components"]

            print(f"Start Request for {REGION} on the {dt}")

            # Extract O3 values and append them to results list
            air_pollution_results.append({
                "Date": dt,
                "O3": air_values.get("o3")
            })

    # 2. case: there is a problem and we give error details
    else:
        print(f"Error message: {response.status_code}")
        print(response.text)


# Looping through the years
years = [2020,2021,2022,2023,2024]

for year in years:
    print(f"collecting air Data for {year}...")

    # § LLM Help for setting up the time dates and converting them into Unix-Timestamp
    start_dt = datetime(year, 1, 1, 0, 0)       
    end_dt = datetime(year, 12, 31, 23, 59)

    start_time = int(start_dt.timestamp())  
    end_time = int(end_dt.timestamp())

    get_air_pollution_data(start_time, end_time)

    # pause to not overload the API
    time.sleep(1) 

# # § LLM Help for creating our dataframe
df = pd.DataFrame(air_pollution_results)
df_daily = df.groupby("Date").mean().reset_index()    
df_daily.to_csv("berlin_o3_2020_2024.csv", index=False)

print("\nData saved as: berlin_o3_2020_2024.csv")
print(df_daily.head())
print("------DONE------")

collecting air Data for 2020...
Start Request for Berlin on the 2020-11-25
Start Request for Berlin on the 2020-11-25
Start Request for Berlin on the 2020-11-25
Start Request for Berlin on the 2020-11-25
Start Request for Berlin on the 2020-11-25
Start Request for Berlin on the 2020-11-25
Start Request for Berlin on the 2020-11-25
Start Request for Berlin on the 2020-11-25
Start Request for Berlin on the 2020-11-25
Start Request for Berlin on the 2020-11-25
Start Request for Berlin on the 2020-11-25
Start Request for Berlin on the 2020-11-25
Start Request for Berlin on the 2020-11-25
Start Request for Berlin on the 2020-11-25
Start Request for Berlin on the 2020-11-25
Start Request for Berlin on the 2020-11-25
Start Request for Berlin on the 2020-11-25
Start Request for Berlin on the 2020-11-25
Start Request for Berlin on the 2020-11-25
Start Request for Berlin on the 2020-11-25
Start Request for Berlin on the 2020-11-25
Start Request for Berlin on the 2020-11-25
Start Request for Berl

In [ ]:
# merging the two csv files
import pandas as pd

# reading the files
df_bird_counts = pd.read_csv(r"C:\Users\ThinkPad\Documents\Data Science Projekt\pigeons_in_berlin.csv") 
df_pollution = pd.read_csv(r"C:\Users\ThinkPad\Documents\Data Science Projekt\berlin_o3_2020_2024.csv")

# § LLM Help for converting dates in both csv's to uniform date format
df_bird_counts["Date"] = pd.to_datetime(df_bird_counts["Date"])    
df_pollution["Date"] = pd.to_datetime(df_pollution["Date"])

# § LLM Help for merging the two files on Date
df_master = pd.merge(df_bird_counts, df_pollution, on="Date", how="inner")    

df_master.to_csv("berlin_pigeon_pullution_2020_2024.csv")
print("csv merged!")

csv merged!
